# 01 — Análisis Exploratorio de Datos
**Criterio 3.1.5.I.1 (10%):** Definición del problema, audiencia, unidad de análisis y KPIs accionables.

**Pregunta central:** ¿Cómo se relaciona el presupuesto de inversión del MOP con el volumen
y tipo de compras registradas en Mercado Público durante 2018-2019?

- **Audiencia:** Directivos de adquisiciones MOP y Contraloría.
- **Unidad de análisis:** Orden de Compra por modalidad, servicio, región y año.
- **Período:** 2018-2019

In [9]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")
pd.set_option("display.float_format", "{:,.1f}".format)

BASE = Path("../01_datos/procesados")
COLS = ["TipoCompra","codigoOC","Anio","Mes","MontoNetoItemCLP",
        "Region","Servicio","Subtitulo","ProveedorRUT","CodBIP","NombreProyecto","Proy_Gasto_MM"]

print("Cargando datos...")
it  = pd.read_csv(BASE / "compras_presupuesto_items.csv", sep=";", low_memory=False, usecols=COLS)
res = pd.read_csv(BASE / "compras_presupuesto_resumen.csv", sep=";", encoding="utf-8-sig")
pres= pd.read_csv(BASE / "presupuesto_2018_2019_unificado.csv", sep=";", encoding="utf-8-sig")

it  = it[it["MontoNetoItemCLP"] > 0].copy()
it["MM"] = it["MontoNetoItemCLP"] / 1e6
it["Trimestre"] = ((it["Mes"]-1)//3+1).map({1:"Q1",2:"Q2",3:"Q3",4:"Q4"})
it["EsInversion"] = (it["Subtitulo"].fillna(-1).astype(int) == 31).astype(int)
it["TieneBIP"] = it["CodBIP"].notna().astype(int)

print(f"Items  : {len(it):,} filas")
print(f"Resumen: {len(res):,} filas")
print(f"Presup.: {len(pres):,} filas")

Cargando datos...
Items  : 172,310 filas
Resumen: 357 filas
Presup.: 3,179 filas


## 1.1 Descripción General de Fuentes

In [10]:
fuentes = pd.DataFrame({
    "Archivo"   : ["compras_presupuesto_items.csv",
                   "compras_presupuesto_resumen.csv",
                   "presupuesto_2018_2019_unificado.csv"],
    "Fuente"    : ["MercadoPublico.cl", "MercadoPublico.cl (agregado)", "MOP Transparencia Activa"],
    "Filas"     : [f"{len(it):,}", f"{len(res):,}", f"{len(pres):,}"],
    "Columnas"  : [str(it.shape[1]), str(res.shape[1]), str(pres.shape[1])],
    "Nivel"     : ["Ítem de OC", "Servicio x Año", "Proyecto BIP"]
})
display(fuentes.set_index("Archivo").style.set_caption("Tabla 1. Fuentes del proyecto"))

,Fuente,Filas,Columnas,Nivel
Archivo,,,,
compras_presupuesto_items.csv,MercadoPublico.cl,"172,310",16,Ítem de OC
compras_presupuesto_resumen.csv,MercadoPublico.cl (agregado),357,13,Servicio x Año
presupuesto_2018_2019_unificado.csv,MOP Transparencia Activa,"3,179",11,Proyecto BIP


## 1.2 KPIs Accionables

In [ ]:
anual  = res.groupby("Anio")[["RS_Presupuesto_MM","RS_Gasto_MM","Compras_MM"]].sum()
n_oc   = it["codigoOC"].nunique()
n_prov = it["ProveedorRUT"].nunique()
tot_MM = it["MM"].sum()

kpis = pd.DataFrame({
    "KPI": ["Órdenes de Compra totales","Proveedores únicos","Monto transado (MM$)",
            "Presupuesto total 2018-2019 (MM$)","Gasto total MOP+MINVU (MM$)","Cobertura Mercado Público (%)"],
    "Valor": [f"{n_oc:,}", f"{n_prov:,}", f"${tot_MM:,.0f}",
              f"${anual.RS_Presupuesto_MM.sum():,.0f}",
              f"${anual.RS_Gasto_MM.sum():,.0f}",
              f"{100*tot_MM/anual.RS_Gasto_MM.sum():.1f}%"],
    "Accionabilidad": [
        "Volumen de contrataciones procesables digitalmente",
        "Diversificación del ecosistema de proveedores",
        "Inversión pública canalizada vía plataforma",
        "Techo presupuestario institucional",
        "Gasto efectivo trazable",
        "Fracción del gasto con trazabilidad digital"
    ]
})
display(kpis.set_index("KPI").style.set_caption("Tabla 2. KPIs accionables"))

## 1.3 Distribución por Modalidad

In [ ]:
tipo = it.groupby("TipoCompra").agg(
    OC=("codigoOC","nunique"), MM=("MM","sum")
).assign(pct_OC=lambda d: 100*d.OC/d.OC.sum(),
         pct_MM=lambda d: 100*d.MM/d.MM.sum()).sort_values("MM", ascending=False)

display(tipo.style
        .format({"OC":"{:,.0f}","MM":"${:,.0f}","pct_OC":"{:.1f}%","pct_MM":"{:.1f}%"})
        .set_caption("Tabla 3. Distribución por modalidad"))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
tipo["MM"].plot.bar(ax=axes[0], color=["#0F4761","#EE0000","#7BA7BC"])
axes[0].set_title("Monto total por modalidad (MM$)"); axes[0].set_ylabel("MM CLP")
axes[0].tick_params(axis="x", rotation=30)
tipo["OC"].plot.pie(ax=axes[1], autopct="%1.1f%%",
                    colors=["#0F4761","#EE0000","#7BA7BC"],
                    labels=tipo.index, startangle=90)
axes[1].set_title("% OC por modalidad"); axes[1].set_ylabel("")
plt.tight_layout()
plt.savefig("../05_figuras/nb01_fig1_modalidades.png", dpi=120, bbox_inches="tight")
plt.show()

## 1.4 Calidad de Datos

In [ ]:
nulos = pd.DataFrame({
    "Columna"  : COLS,
    "Nulos"    : [it[c].isna().sum() for c in COLS],
    "% Nulo"   : [f"{100*it[c].isna().mean():.1f}%" for c in COLS],
    "Tipo"     : [str(it[c].dtype) for c in COLS],
    "Únicos"   : [it[c].nunique() for c in COLS]
})
display(nulos.set_index("Columna").style.set_caption("Tabla 4. Calidad de datos por columna"))

## 1.5 Distribución Anual

In [ ]:
anual_tipo = it.groupby(["Anio","TipoCompra"]).MM.sum().unstack(fill_value=0)
display(anual_tipo.style
        .format("${:,.0f}")
        .set_caption("Tabla 5. Monto (MM$) por año y modalidad"))

fig, ax = plt.subplots(figsize=(8, 4))
anual_tipo.plot.bar(ax=ax, color=["#0F4761","#EE0000","#7BA7BC"], width=0.6)
ax.set_title("Compras por año y modalidad (MM$)")
ax.set_ylabel("MM CLP"); ax.tick_params(axis="x", rotation=0)
plt.tight_layout()
plt.savefig("../05_figuras/nb01_fig2_anual.png", dpi=120, bbox_inches="tight")
plt.show()